In [1]:
#Intention Detection based on the Subcat method

In [24]:
from __future__ import annotations

import math
import re
from pathlib import Path
from typing import Dict, List, Tuple, Set, Optional
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer

# -------------------------
# Stopwords (for n-gram suggestions)
# -------------------------
try:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    STOP: Set[str] = set(ENGLISH_STOP_WORDS)
except Exception:
    STOP = {
        "the", "a", "an", "and", "or", "to", "of", "in", "on", "for", "with", "by",
        "is", "are", "was", "were", "be", "this", "that", "it", "as", "at", "from"
    }


# ============================================================
# I/O (your paths)
# ============================================================
BASE_DIR = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention"
)

EPISODES_CSV = BASE_DIR / "2_All_episodes_with_messages.csv"
COMMITS_CSV = BASE_DIR / "1_All_Commits_PR_Msg_Iss.csv"

# Use the boosted dictionary WITHOUT blacklist
DICTIONARY_CSV = BASE_DIR / "0_dictionary_lemmatized.csv"

OUTPUT_DIR = BASE_DIR / "Output"
OUT_EPISODES = OUTPUT_DIR / "2_All_episodes_with_intentions_subcat_tfidf.csv"
OUT_COMMITS = OUTPUT_DIR / "1_All_Commits_PR_Msg_Iss_with_intentions_subcat_tfidf.csv"

# Candidate suggestion outputs (like your attached example)
OUT_CANDIDATES = OUTPUT_DIR / "dictionary_candidate_suggestions_ngrams_idf.csv"


# ============================================================
# Context weights
# ============================================================
ISSUE_BOOST = 1.5
COMMIT_BOOST = 1.0
PR_BOOST = 1.0


# ============================================================
# Scoring / selection params (TF-IDF scale)
# ============================================================
MIN_SCORE = 0.12
MULTI_RATIO = 0.80
MAX_LABELS = 3

CONF_W_SHARE = 0.6
CONF_W_MARGIN = 0.4

HIGH_TH = 0.75
MED_TH = 0.55


# ============================================================
# Episode schema
# ============================================================
END_COMMIT_COL = "episode_end_commit_sha"


# ============================================================
# Column specs (episodes + commits)
# Note: also includes issue summaries if present (optional)
# ============================================================
EP_START_PARTS: List[Tuple[List[str], float]] = [
    (["start_commit_subject", "start_commit_body"], COMMIT_BOOST),

    (["start_pr_titles"], PR_BOOST),
    (["start_pr_bodies"], PR_BOOST),
    (["start_pr_comments_and_reviews"], PR_BOOST),

    (["start_issue_titles"], ISSUE_BOOST),
    (["start_issue_bodies"], ISSUE_BOOST),
    (["start_issue_comments"], ISSUE_BOOST),

    # optional
    (["start_issue_summary"], ISSUE_BOOST),
]

EP_END_PARTS: List[Tuple[List[str], float]] = [
    (["end_commit_subject", "end_commit_body"], COMMIT_BOOST),

    (["end_pr_titles"], PR_BOOST),
    (["end_pr_bodies"], PR_BOOST),
    (["end_pr_comments_and_reviews"], PR_BOOST),

    (["end_issue_titles"], ISSUE_BOOST),
    (["end_issue_bodies"], ISSUE_BOOST),
    (["end_issue_comments"], ISSUE_BOOST),

    # optional
    (["end_issue_summary"], ISSUE_BOOST),
]

COMMIT_PARTS: List[Tuple[List[str], float]] = [
    (["commit_subject", "commit_body"], COMMIT_BOOST),

    (["pr_titles"], PR_BOOST),
    (["pr_bodies"], PR_BOOST),
    (["pr_comments_and_reviews"], PR_BOOST),

    (["issue_titles"], ISSUE_BOOST),
    (["issue_bodies"], ISSUE_BOOST),
    (["issue_comments"], ISSUE_BOOST),
]


# ============================================================
# Helpers
# ============================================================
def _safe_str(v) -> str:
    if v is None:
        return ""
    if isinstance(v, float) and pd.isna(v):
        return ""
    s = str(v).strip()
    return "" if s.lower() == "nan" else s


def normalize_text(text: str) -> str:
    # de-camelcase + normalize separators
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", str(text))
    text = text.replace("_", " ").replace("-", " ")
    return text.lower()


# ---- Optional stemming ----
try:
    from nltk.stem.snowball import SnowballStemmer
    _stemmer = SnowballStemmer("english")

    def _stem(token: str) -> str:
        return _stemmer.stem(token)
except Exception:
    def _stem(token: str) -> str:
        return token


def tokenize_and_stem(text: str) -> List[str]:
    """
    Tokenize into [a-z0-9]+ and stem alphabetic tokens.
    Keeps digits (e2e, v2, etc).
    """
    if not text:
        return []
    text = normalize_text(text)
    raw = re.findall(r"[a-z0-9]+", text)
    out: List[str] = []
    for t in raw:
        if any(ch.isdigit() for ch in t):
            out.append(t)
        else:
            out.append(_stem(t))
    return out


def build_parts_from_row(row: pd.Series, spec: List[Tuple[List[str], float]]) -> List[Tuple[str, float]]:
    parts: List[Tuple[str, float]] = []
    for cols, mult in spec:
        # include only columns that exist
        texts = []
        for c in cols:
            if c in row.index:
                texts.append(_safe_str(row.get(c, "")))
        txt = "\n".join(texts).strip()
        if txt:
            parts.append((txt, float(mult)))
    return parts


def boundary_has_end_commit(row: pd.Series) -> bool:
    end_sha = _safe_str(row.get(END_COMMIT_COL, ""))
    return bool(end_sha.strip())


def join_parts(parts: List[Tuple[str, float]]) -> str:
    return "\n".join([t for (t, _m) in parts]).strip()


# ============================================================
# Dictionary loading (LONG format) - blacklist DROPPED
# ============================================================
def load_dictionary_long(path: Path) -> Dict[str, List[Dict]]:
    """
    Expects columns: label, keyword, optional weight, optional group.
    Any rows with label == 'Blacklist' are ignored (dropped).
    """
    df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8", engine="python")
    colmap = {str(c).strip().lower(): c for c in df.columns}

    if "label" not in colmap or "keyword" not in colmap:
        raise ValueError("Dictionary CSV must have columns: label, keyword")

    label_col = colmap["label"]
    keyword_col = colmap["keyword"]
    weight_col = colmap.get("weight", None)
    group_col = colmap.get("group", None)

    dict_terms: Dict[str, List[Dict]] = defaultdict(list)

    for _, row in df.iterrows():
        lab = _safe_str(row.get(label_col, "")).strip()
        kw = _safe_str(row.get(keyword_col, "")).strip()
        if not lab or not kw:
            continue
        if lab.lower() == "blacklist":
            continue  # explicitly dropped

        wt = 1.0
        if weight_col is not None:
            try:
                wt = float(row.get(weight_col, 1.0))
            except Exception:
                wt = 1.0

        grp = _safe_str(row.get(group_col, "")).strip().upper() if group_col is not None else ""
        dict_terms[lab].append({"keyword": kw, "weight": float(wt), "group": grp})

    if not dict_terms:
        raise ValueError("No label keywords loaded from dictionary CSV")

    # Deduplicate per label by keyword string
    out: Dict[str, List[Dict]] = {}
    for lab, items in dict_terms.items():
        seen = set()
        deduped = []
        for it in items:
            k = it["keyword"].strip().lower()
            if k in seen:
                continue
            seen.add(k)
            deduped.append(it)
        out[lab] = deduped

    return out


def build_dictionary_phrase_set(dict_terms: Dict[str, List[Dict]]) -> Set[str]:
    """
    Phrases in the same token space as candidates:
      phrase = " ".join(tokenize_and_stem(keyword))
    """
    s: Set[str] = set()
    for _lab, items in dict_terms.items():
        for it in items:
            toks = tokenize_and_stem(it["keyword"])
            if toks:
                s.add(" ".join(toks))
    return s


# ============================================================
# Constraints using GLOBAL groups
# ============================================================
def passes_constraints(label: str, global_groups: Set[str]) -> bool:
    if label == "Introduce / strengthen CI-backed tests":
        return ("CI" in global_groups) and ("TEST" in global_groups)

    if label == "Migrate or modernise CI infrastructure":
        return ("MIGRATE" in global_groups) and ("CI" in global_groups)

    if label == "Clean up or simplify CI / environment configuration":
        return ("CLEANUP" in global_groups) and (("CI" in global_groups) or ("ENV" in global_groups))

    # Perf/Stability only when stability signal exists + CI/TEST context exists
    if label == "Address performance or stability issues":
        return (("PERF" in global_groups) or ("FAIL" in global_groups)) and (
            ("CI" in global_groups) or ("TEST" in global_groups)
        )

    return True


def apply_precedence(scores: Dict[str, float], matched_keywords: Dict[str, List[str]], global_groups: Set[str]) -> None:
    """
    Prefer Perf/Stability over CI-backed-tests when both stability + CI/TEST signals are present.
    """
    perf_lab = "Address performance or stability issues"
    ci_lab = "Introduce / strengthen CI-backed tests"

    has_stability_signal = ("PERF" in global_groups) or ("FAIL" in global_groups)
    has_ci_or_test = ("CI" in global_groups) or ("TEST" in global_groups)

    if has_stability_signal and has_ci_or_test:
        # ---- FIX: guard against KeyError if ci_lab not present in dictionary ----
        if ci_lab in scores:
            scores[ci_lab] = 0.0
            matched_keywords[ci_lab] = []
        if scores.get(perf_lab, 0.0) > 0:
            scores[perf_lab] *= 1.5


# ============================================================
# Selection + confidence
# ============================================================
def assign_labels_multi(
    scores: Dict[str, float],
    matched_keywords: Dict[str, List[str]],
    min_score: float = MIN_SCORE,
    multi_ratio: float = MULTI_RATIO,
    max_labels: int = MAX_LABELS,
) -> Dict[str, object]:
    items = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    top_label, top_score = items[0] if items else ("", 0.0)
    second_label, second_score = items[1] if len(items) > 1 else ("", 0.0)

    total = float(sum(scores.values()))
    if top_score < min_score:
        return {
            "label_str": None,
            "top_label": top_label, "top_score": float(top_score),
            "second_label": second_label, "second_score": float(second_score),
            "total_score": float(total),
            "selected_score_sum": 0.0,
            "confidence_share_selected": 0.0,
            "confidence_margin_selected": 0.0,
            "confidence_value": 0.0,
            "confidence_level": "UNLABELED",
            "matched_keywords": "",
            "suggested_ngrams": "",
        }

    chosen: List[str] = []
    chosen_scores: List[float] = []

    for lab, sc in items:
        if sc < min_score:
            break
        if sc >= top_score * multi_ratio:
            chosen.append(lab)
            chosen_scores.append(sc)
        if len(chosen) >= max_labels:
            break

    chosen_set = set(chosen)
    next_unselected = 0.0
    for lab, sc in items:
        if lab not in chosen_set:
            next_unselected = sc
            break

    selected_sum = float(sum(chosen_scores))
    share_selected = (selected_sum / total) if total > 0 else 0.0
    min_selected = float(min(chosen_scores)) if chosen_scores else 0.0
    margin_selected = ((min_selected - next_unselected) / min_selected) if min_selected > 0 else 0.0
    margin_selected = max(0.0, min(1.0, margin_selected))

    confidence_value = (CONF_W_SHARE * share_selected) + (CONF_W_MARGIN * margin_selected)

    if confidence_value >= HIGH_TH and selected_sum >= (min_score * 2.0):
        level = "HIGH"
    elif confidence_value >= MED_TH:
        level = "MEDIUM"
    else:
        level = "LOW"

    matched = sorted({kw for lab in chosen for kw in matched_keywords.get(lab, [])})

    return {
        "label_str": " || ".join(chosen),
        "top_label": top_label, "top_score": float(top_score),
        "second_label": second_label, "second_score": float(second_score),
        "total_score": float(total),
        "selected_score_sum": float(selected_sum),
        "confidence_share_selected": float(share_selected),
        "confidence_margin_selected": float(margin_selected),
        "confidence_value": float(confidence_value),
        "confidence_level": level,
        "matched_keywords": "; ".join(matched),
        "suggested_ngrams": "",
    }


# ============================================================
# Safe stability override (near CI/test language)
# ============================================================
PERF_LABEL = "Address performance or stability issues"

NEAR_TERMS = {
    "test", "tests", "ci", "build", "pipeline", "workflow",
    "instrument", "instrumentation", "github", "actions",
}

STRONG_STAB_PHRASES = [
    "flaky", "flakiness", "flake", "intermittent", "nondeterministic",
    "race condition", "deadlock", "hang", "hung",
    "quarantine", "quarantine test", "disable test", "disabled test",
    "failing test", "fix flaky", "fix flake",
]

WEAK_STAB_PHRASES = [
    "timeout", "timeouts", "timed out", "timing out",
    "retry", "retries",
]


def _phrase_positions(tokens: List[str], phrase_tokens: List[str]) -> List[int]:
    if not phrase_tokens or not tokens:
        return []
    if len(phrase_tokens) == 1:
        p = phrase_tokens[0]
        return [i for i, t in enumerate(tokens) if t == p]
    n = len(phrase_tokens)
    out = []
    for i in range(len(tokens) - n + 1):
        if tokens[i: i + n] == phrase_tokens:
            out.append(i)
    return out


def stability_context_boost(text: str, window: int = 5) -> float:
    """
    Add a small boost to PERF_LABEL only when stability keywords appear
    near CI/test language.
    """
    toks = tokenize_and_stem(text)
    if not toks:
        return 0.0

    near_positions = {i for i, t in enumerate(toks) if t in NEAR_TERMS}
    if not near_positions:
        return 0.0

    boost = 0.0

    def apply(phrases: List[str], amount: float) -> None:
        nonlocal boost
        for ph in phrases:
            ph_toks = tokenize_and_stem(ph)
            for pos in _phrase_positions(toks, ph_toks):
                if any(abs(pos - j) <= window for j in near_positions):
                    boost += amount
                    return  # count each strength group at most once

    apply(STRONG_STAB_PHRASES, 0.20)
    apply(WEAK_STAB_PHRASES, 0.10)

    return min(boost, 0.40)


# ============================================================
# TF-IDF dictionary scorer (no blacklist)
# ============================================================
class TfidfDictionaryScorer:
    def __init__(self, dict_terms: Dict[str, List[Dict]]) -> None:
        self.dict_terms = dict_terms

        self.labels: List[str] = sorted(dict_terms.keys())
        self.label_to_i = {lab: i for i, lab in enumerate(self.labels)}

        self.vectorizer: Optional[TfidfVectorizer] = None
        self.vocab_size: int = 0

        self.label_matrix: Optional[csr_matrix] = None

        # feature -> groups (for constraints)
        self.groups_by_feature: Dict[int, Set[str]] = defaultdict(set)

        # label -> feature -> keywords (for matched keyword lists)
        self.label_feature_to_keywords: Dict[str, Dict[int, List[str]]] = defaultdict(lambda: defaultdict(list))

    def _all_dictionary_texts(self) -> List[str]:
        texts = []
        for _lab, items in self.dict_terms.items():
            for it in items:
                texts.append(it["keyword"])
        return texts

    def fit(self, corpus_texts: List[str]) -> None:
        self.vectorizer = TfidfVectorizer(
            analyzer="word",
            tokenizer=tokenize_and_stem,
            preprocessor=normalize_text,
            token_pattern=None,
            ngram_range=(1, 3),
            lowercase=False,
            norm="l2",
            smooth_idf=True,
            sublinear_tf=False,
        )

        fit_texts = corpus_texts + self._all_dictionary_texts()
        self.vectorizer.fit(fit_texts)

        self.vocab_size = len(self.vectorizer.vocabulary_)
        self._build_label_matrix_and_maps()

    def _phrase_to_feature_index(self, phrase: str) -> Optional[int]:
        assert self.vectorizer is not None
        toks = tokenize_and_stem(phrase)
        if not toks:
            return None
        feat = " ".join(toks)
        return self.vectorizer.vocabulary_.get(feat)

    def _build_label_matrix_and_maps(self) -> None:
        assert self.vectorizer is not None

        rows: List[int] = []
        cols: List[int] = []
        data: List[float] = []

        for lab, items in self.dict_terms.items():
            r = self.label_to_i[lab]
            for it in items:
                idx = self._phrase_to_feature_index(it["keyword"])
                if idx is None:
                    continue

                rows.append(r)
                cols.append(idx)
                data.append(float(it.get("weight", 1.0)))

                grp = (it.get("group") or "").strip().upper()
                if grp:
                    self.groups_by_feature[idx].add(grp)

                self.label_feature_to_keywords[lab][idx].append(it["keyword"])

        self.label_matrix = csr_matrix((data, (rows, cols)), shape=(len(self.labels), self.vocab_size))

    def transform_text(self, text: str) -> csr_matrix:
        assert self.vectorizer is not None
        return self.vectorizer.transform([text])

    def score_parts(
        self,
        parts: List[Tuple[str, float]],
    ) -> Tuple[Dict[str, float], Dict[str, List[str]], Set[str]]:
        """
        Returns:
          scores[label] = TF-IDF-weighted dictionary sum
          matched_keywords[label] = list[str]
          global_groups = set[str]
        """
        assert self.label_matrix is not None

        scores_vec = np.zeros(len(self.labels), dtype=float)
        boundary_feature_hits: Set[int] = set()

        for text, mult in parts:
            v = self.transform_text(text)
            if v.nnz == 0:
                continue

            contrib = (self.label_matrix @ v.T).toarray().ravel()
            scores_vec += contrib * float(mult)

            boundary_feature_hits.update(v.indices)

        scores: Dict[str, float] = {lab: float(scores_vec[self.label_to_i[lab]]) for lab in self.labels}

        # global groups = union of groups for any feature hit
        global_groups: Set[str] = set()
        for fi in boundary_feature_hits:
            global_groups |= self.groups_by_feature.get(fi, set())

        # matched keywords
        matched_keywords: Dict[str, List[str]] = {lab: [] for lab in self.labels}
        for lab in self.labels:
            feats = self.label_feature_to_keywords.get(lab, {})
            hits: Set[str] = set()
            for fi in boundary_feature_hits:
                if fi in feats:
                    hits.update(feats[fi])
            matched_keywords[lab] = sorted(hits)

        return scores, matched_keywords, global_groups


# ============================================================
# Candidate n-gram suggestion logic (like your attached CSV)
# Score = tf_weighted * ln((N+1)/(df+1))   [NO +1 added to the log result]
# ============================================================
def candidate_counter_for_parts(parts: List[Tuple[str, float]], dict_phrase_set: Set[str], max_n: int = 3) -> Counter[str]:
    """
    Produce weighted counts of candidate 1–3 grams for ONE boundary doc, excluding dictionary phrases.
    - Uses stemmed token space.
    - Drops stopwords from token stream for candidate generation.
    - WEIGHTS each occurrence by the part multiplier (commit/pr/issue boost), so output tf_weighted is meaningful.
    """
    c: Counter[str] = Counter()

    for text, mult in parts:
        toks0 = tokenize_and_stem(text)
        toks = [t for t in toks0 if t and (t not in STOP)]
        L = len(toks)
        if L == 0:
            continue

        w = float(mult)

        for n in range(1, max_n + 1):
            if n > L:
                continue

            if n == 1:
                for t in toks:
                    phrase = t
                    if phrase in dict_phrase_set:
                        continue
                    c[phrase] += w
            else:
                for i in range(L - n + 1):
                    ng = toks[i: i + n]
                    if all(wd in STOP for wd in ng):
                        continue
                    phrase = " ".join(ng)
                    if phrase in dict_phrase_set:
                        continue
                    c[phrase] += w

    return c


def idf_like(N: int, df: int) -> float:
    # match your attached file: ln((N+1)/(df+1))  (no "+1" added after log)
    return math.log((N + 1.0) / (df + 1.0))


# ============================================================
# Build fitting corpus (episodes + commits)
# ============================================================
def build_fit_corpus_for_vectorizer(ep_df: pd.DataFrame, cm_df: pd.DataFrame) -> List[str]:
    corpus: List[str] = []

    for _, row in ep_df.iterrows():
        p = build_parts_from_row(row, EP_START_PARTS)
        t = join_parts(p)
        if t:
            corpus.append(t)

        if boundary_has_end_commit(row):
            pe = build_parts_from_row(row, EP_END_PARTS)
            te = join_parts(pe)
            if te:
                corpus.append(te)

    for _, row in cm_df.iterrows():
        p = build_parts_from_row(row, COMMIT_PARTS)
        t = join_parts(p)
        if t:
            corpus.append(t)

    return corpus


# ============================================================
# Label one boundary/row + attach suggested_ngrams later
# ============================================================
def label_boundary(row: pd.Series, parts_spec, scorer: TfidfDictionaryScorer) -> Dict[str, object]:
    parts = build_parts_from_row(row, parts_spec)
    text_joined = join_parts(parts)

    scores, matched_keywords, global_groups = scorer.score_parts(parts)

    # Safe stability override
    scores[PERF_LABEL] = scores.get(PERF_LABEL, 0.0) + stability_context_boost(text_joined)

    # Constraints
    for lab in list(scores.keys()):
        if not passes_constraints(lab, global_groups):
            scores[lab] = 0.0
            matched_keywords[lab] = []

    # Precedence
    apply_precedence(scores, matched_keywords, global_groups)

    return assign_labels_multi(scores, matched_keywords)


def _warn_missing_cols(df: pd.DataFrame, required: List[str], name: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"[warn] {name}: missing expected columns:")
        for c in missing:
            print("  -", c)


# ============================================================
# Main
# ============================================================
def main() -> None:
    if not EPISODES_CSV.exists():
        raise FileNotFoundError(f"Episodes CSV not found: {EPISODES_CSV}")
    if not COMMITS_CSV.exists():
        raise FileNotFoundError(f"Commits CSV not found: {COMMITS_CSV}")
    if not DICTIONARY_CSV.exists():
        raise FileNotFoundError(f"Dictionary CSV not found: {DICTIONARY_CSV}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    ep = pd.read_csv(EPISODES_CSV, dtype=str, keep_default_na=False, encoding="utf-8", engine="python")
    cm = pd.read_csv(COMMITS_CSV, dtype=str, keep_default_na=False, encoding="utf-8", engine="python")

    _warn_missing_cols(
        ep,
        required=[
            END_COMMIT_COL,
            "start_commit_subject", "start_commit_body",
            "start_pr_titles", "start_pr_bodies", "start_pr_comments_and_reviews",
            "start_issue_titles", "start_issue_bodies", "start_issue_comments",
            "end_commit_subject", "end_commit_body",
            "end_pr_titles", "end_pr_bodies", "end_pr_comments_and_reviews",
            "end_issue_titles", "end_issue_bodies", "end_issue_comments",
        ],
        name="EPISODES_CSV",
    )

    _warn_missing_cols(
        cm,
        required=[
            "repo_name", "commit_sha",
            "commit_subject", "commit_body",
            "pr_titles", "pr_bodies", "pr_comments_and_reviews",
            "issue_titles", "issue_bodies", "issue_comments",
        ],
        name="COMMITS_CSV",
    )

    # Load dictionary + build phrase set for candidate filtering
    dict_terms = load_dictionary_long(DICTIONARY_CSV)
    dict_phrase_set = build_dictionary_phrase_set(dict_terms)

    # Fit TF-IDF (for intention detection)
    corpus = build_fit_corpus_for_vectorizer(ep, cm)
    scorer = TfidfDictionaryScorer(dict_terms)
    scorer.fit(corpus)

    # ----------------------------
    # 1) Intention detection (episodes)
    # ----------------------------
    start_results: List[Dict[str, object]] = []
    end_results: List[Dict[str, object]] = []

    # We'll also store per-doc counters for keyword suggestions
    start_doc_counters: List[Counter[str]] = []
    end_doc_counters: List[Counter[str]] = []

    # Aggregate candidate stats (like your attached file)
    agg_tf: Dict[str, Counter[str]] = {"start": Counter(), "end": Counter(), "commit": Counter()}
    agg_df: Dict[str, Counter[str]] = {"start": Counter(), "end": Counter(), "commit": Counter()}
    agg_example: Dict[str, Dict[str, Dict[str, str]]] = {"start": {}, "end": {}, "commit": {}}
    doc_count_N: Dict[str, int] = {"start": 0, "end": 0, "commit": 0}

    # --- Episodes loop ---
    for _, row in ep.iterrows():
        # start boundary
        s = label_boundary(row, EP_START_PARTS, scorer)
        start_results.append(s)

        s_parts = build_parts_from_row(row, EP_START_PARTS)
        if s_parts:
            doc_count_N["start"] += 1
            c = candidate_counter_for_parts(s_parts, dict_phrase_set, max_n=3)
            start_doc_counters.append(c)

            if c:
                agg_df["start"].update(set(c.keys()))
                agg_tf["start"].update(c)

                repo_name = _safe_str(row.get("repo_name", ""))
                ep_index = _safe_str(row.get("episode_index", ""))
                for phrase in c.keys():
                    if phrase not in agg_example["start"]:
                        agg_example["start"][phrase] = {
                            "example_repo_name": repo_name,
                            "example_episode_index": ep_index,
                            "example_commit_sha": "",
                        }
        else:
            start_doc_counters.append(Counter())

        # end boundary
        if not boundary_has_end_commit(row):
            end_results.append({
                "label_str": None,
                "top_label": "", "top_score": 0.0,
                "second_label": "", "second_score": 0.0,
                "total_score": 0.0,
                "selected_score_sum": 0.0,
                "confidence_share_selected": 0.0,
                "confidence_margin_selected": 0.0,
                "confidence_value": 0.0,
                "confidence_level": "NO_END_COMMIT",
                "matched_keywords": "",
                "suggested_ngrams": "",
            })
            end_doc_counters.append(Counter())
        else:
            e = label_boundary(row, EP_END_PARTS, scorer)
            end_results.append(e)

            e_parts = build_parts_from_row(row, EP_END_PARTS)
            if e_parts:
                doc_count_N["end"] += 1
                c = candidate_counter_for_parts(e_parts, dict_phrase_set, max_n=3)
                end_doc_counters.append(c)

                if c:
                    agg_df["end"].update(set(c.keys()))
                    agg_tf["end"].update(c)

                    repo_name = _safe_str(row.get("repo_name", ""))
                    ep_index = _safe_str(row.get("episode_index", ""))
                    for phrase in c.keys():
                        if phrase not in agg_example["end"]:
                            agg_example["end"][phrase] = {
                                "example_repo_name": repo_name,
                                "example_episode_index": ep_index,
                                "example_commit_sha": "",
                            }
            else:
                end_doc_counters.append(Counter())

    # ----------------------------
    # 2) Intention detection (commits)
    # ----------------------------
    cm_results: List[Dict[str, object]] = []
    commit_doc_counters: List[Counter[str]] = []

    for _, row in cm.iterrows():
        r = label_boundary(row, COMMIT_PARTS, scorer)
        cm_results.append(r)

        c_parts = build_parts_from_row(row, COMMIT_PARTS)
        if c_parts:
            doc_count_N["commit"] += 1
            c = candidate_counter_for_parts(c_parts, dict_phrase_set, max_n=3)
            commit_doc_counters.append(c)

            if c:
                agg_df["commit"].update(set(c.keys()))
                agg_tf["commit"].update(c)

                repo_name = _safe_str(row.get("repo_name", ""))
                sha = _safe_str(row.get("commit_sha", ""))
                for phrase in c.keys():
                    if phrase not in agg_example["commit"]:
                        agg_example["commit"][phrase] = {
                            "example_repo_name": repo_name,
                            "example_episode_index": "",
                            "example_commit_sha": sha,
                        }
        else:
            commit_doc_counters.append(Counter())

    # ============================================================
    # Suggested n-grams per row (top_k by tf_weighted * idf_like)
    # ============================================================
    def top_doc_suggestions(boundary: str, doc_counter: Counter[str], top_k: int = 20) -> str:
        if not doc_counter:
            return ""
        N = doc_count_N.get(boundary, 0)
        if N <= 0:
            return ""
        scored = []
        for phrase, tfw in doc_counter.items():
            df_docs = int(agg_df[boundary].get(phrase, 0))
            score = float(tfw) * idf_like(N, df_docs)
            if score > 0:
                scored.append((phrase, score))
        scored.sort(key=lambda x: x[1], reverse=True)
        return "; ".join([p for p, _ in scored[:top_k]])

    for i in range(len(start_results)):
        start_results[i]["suggested_ngrams"] = top_doc_suggestions("start", start_doc_counters[i], top_k=20)

    for i in range(len(end_results)):
        if end_results[i].get("confidence_level") == "NO_END_COMMIT":
            continue
        end_results[i]["suggested_ngrams"] = top_doc_suggestions("end", end_doc_counters[i], top_k=20)

    for i in range(len(cm_results)):
        cm_results[i]["suggested_ngrams"] = top_doc_suggestions("commit", commit_doc_counters[i], top_k=20)

    # ============================================================
    # Write episodes + commits outputs
    # ============================================================
    ep_out = pd.concat(
        [ep, pd.DataFrame(start_results).add_prefix("start_"), pd.DataFrame(end_results).add_prefix("end_")],
        axis=1
    )
    ep_out.to_csv(OUT_EPISODES, index=False, encoding="utf-8")
    print("[ok] wrote episodes intentions:", OUT_EPISODES)

    cm_int = pd.DataFrame(cm_results).add_prefix("intent_")
    cm_out = pd.concat([cm, cm_int], axis=1)
    cm_out.to_csv(OUT_COMMITS, index=False, encoding="utf-8")
    print("[ok] wrote commit intentions:", OUT_COMMITS)

    # ============================================================
    # Aggregate candidate suggestions (like attached)
    # Columns:
    # boundary, candidate_phrase, score, tf_weighted, df_docs, doc_count_N, df_ratio,
    # example_repo_name, example_episode_index, example_commit_sha
    # ============================================================
    rows = []
    for boundary in ["start", "end", "commit"]:
        N = doc_count_N.get(boundary, 0)
        if N <= 0:
            continue

        for phrase, tfw in agg_tf[boundary].items():
            df_docs = int(agg_df[boundary].get(phrase, 0))
            if df_docs <= 0:
                continue
            idf = idf_like(N, df_docs)
            score = float(tfw) * idf
            ex = agg_example[boundary].get(phrase, {})
            rows.append({
                "boundary": boundary,
                "candidate_phrase": phrase,
                "score": score,
                "tf_weighted": float(tfw),
                "df_docs": df_docs,
                "doc_count_N": N,
                "df_ratio": (df_docs / float(N)) if N > 0 else 0.0,
                "example_repo_name": ex.get("example_repo_name", ""),
                "example_episode_index": ex.get("example_episode_index", ""),
                "example_commit_sha": ex.get("example_commit_sha", ""),
            })

    cand = pd.DataFrame(rows)
    if not cand.empty:
        cand = cand.sort_values(["boundary", "score"], ascending=[True, False])

        # Keep top 500 per boundary
        cand_out_parts = []
        for boundary in ["start", "end", "commit"]:
            sub = cand[cand["boundary"] == boundary].head(500)
            if not sub.empty:
                cand_out_parts.append(sub)
        cand_out = pd.concat(cand_out_parts, axis=0) if cand_out_parts else cand

        cand_out.to_csv(OUT_CANDIDATES, index=False, encoding="utf-8")
        print("[ok] wrote n-gram dictionary candidate suggestions:", OUT_CANDIDATES)
    else:
        print("[warn] no candidate n-grams were generated (empty output).")

    print("[done] episodes_rows =", len(ep_out), " | commits_rows =", len(cm_out))


if __name__ == "__main__":
    main()


[ok] wrote episodes intentions: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention\Output\2_All_episodes_with_intentions_subcat_tfidf.csv
[ok] wrote commit intentions: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention\Output\1_All_Commits_PR_Msg_Iss_with_intentions_subcat_tfidf.csv
[ok] wrote n-gram dictionary candidate suggestions: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention\Output\dictionary_candidate_suggestions_ngrams_idf.csv
[done] episodes_rows = 535  | commits_rows = 535


In [14]:
#top 20

In [18]:
from __future__ import annotations

import re
from pathlib import Path
import pandas as pd

# ============================================================
# Paths (match your folder layout)
# ============================================================
BASE_DIR = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention"
)
OUTPUT_DIR = BASE_DIR / "Output"

CAND_IN = OUTPUT_DIR / "dictionary_candidate_suggestions_ngrams_idf.csv"
CAND_OUT = OUTPUT_DIR / "dictionary_candidate_suggestions_TOP20_PER_BOUNDARY.csv"

# ============================================================
# Strategy thresholds (tune if needed)
# ============================================================
TOP_K_PER_BOUNDARY = 20

# "Sweet spot" band (good balance between generality + discrimination)
SWEET_MIN = 0.01     # ~1% of boundaries
SWEET_MAX = 0.15     # ~15% of boundaries

# Above this tends to be too generic for dictionary labels
GENERIC_MAX = 0.25   # ~25% of boundaries

# Require some repetition to reduce one-off noise
MIN_DF_DOCS = 5

# ============================================================
# Noise filtering
# ============================================================
STOPWORDS = {
    "a","an","the","and","or","to","of","in","on","for","with","by","from","as","at","it","is","are","be",
    "this","that","these","those","we","you","they","i","our","your","their","was","were","will","can",
    "not","no","yes","if","then","else","when","while","into","about","over","under","up","down","out",
    "more","most","some","any","all","one","two","three","use","using","used"
}

NOISE_TOKENS = {
    "http","https","www","utm","utm_source","utm_medium","utm_campaign","utm_term","utm_content",
    "href","src","png","jpg","jpeg","gif","svg","html","php"
}

_alnum_re = re.compile(r"^[a-z0-9]+$")


def looks_like_noise_phrase(phrase: str) -> bool:
    """
    Conservative filter:
      - drops URL/tracking/artifact tokens
      - drops punctuationy tokens
      - drops stopword-only phrases
      - drops mostly-numeric phrases
    NOTE: Unlike your original version, this now ALLOWS unigrams.
    """
    p = (phrase or "").strip().lower()
    if not p:
        return True

    toks = p.split()
    if not toks:
        return True

    if any(t in NOISE_TOKENS for t in toks):
        return True

    if any(not _alnum_re.match(t) for t in toks):
        return True

    if all(t in STOPWORDS for t in toks):
        return True

    num_ratio = sum(1 for t in toks if t.isdigit()) / len(toks)
    if num_ratio >= 0.67:
        return True

    # allow 1-word phrases
    return False


# ============================================================
# Selection logic: Sweet spot first, then moderate, then rare
# ============================================================
def select_top_k(df: pd.DataFrame, k: int = 20) -> pd.DataFrame:
    # Basic hygiene + types
    for col in ["score", "tf_weighted", "df_docs", "doc_count_N", "df_ratio"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["candidate_phrase", "score", "df_docs", "doc_count_N", "df_ratio"]).copy()
    df["candidate_phrase"] = df["candidate_phrase"].astype(str)
    df["boundary"] = df["boundary"].astype(str)

    # Remove noise
    df = df[~df["candidate_phrase"].map(looks_like_noise_phrase)].copy()

    # Remove too-generic candidates
    df = df[df["df_ratio"] <= GENERIC_MAX].copy()

    # Require repetition
    df = df[df["df_docs"] >= MIN_DF_DOCS].copy()

    # Buckets
    sweet = df[(df["df_ratio"] >= SWEET_MIN) & (df["df_ratio"] <= SWEET_MAX)].copy()
    moderate = df[(df["df_ratio"] > SWEET_MAX) & (df["df_ratio"] <= GENERIC_MAX)].copy()
    rare = df[df["df_ratio"] < SWEET_MIN].copy()

    # Sort within buckets
    sort_cols = ["score", "tf_weighted", "df_docs", "df_ratio"]
    sweet = sweet.sort_values(sort_cols, ascending=[False, False, False, True])
    moderate = moderate.sort_values(sort_cols, ascending=[False, False, False, True])
    rare = rare.sort_values(sort_cols, ascending=[False, False, False, True])

    combined = pd.concat([sweet, moderate, rare], ignore_index=True)

    # De-dupe by phrase *within this boundary* (keep best)
    combined = combined.drop_duplicates(subset=["candidate_phrase"], keep="first")

    return combined.head(k)


def main() -> None:
    if not CAND_IN.exists():
        raise FileNotFoundError(f"Input suggestions file not found: {CAND_IN}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(CAND_IN, dtype=str, keep_default_na=False, encoding="utf-8", engine="python")

    required = {"boundary", "candidate_phrase", "score", "tf_weighted", "df_docs", "doc_count_N", "df_ratio"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {CAND_IN}: {sorted(missing)}")

    # Top-K per boundary (start/end/commit)
    top = (
        df.groupby("boundary", group_keys=False)
          .apply(lambda g: select_top_k(g, k=TOP_K_PER_BOUNDARY))
          .reset_index(drop=True)
    )

    # Keep a clean column order (only include if present)
    cols = [
        "boundary", "candidate_phrase", "score", "tf_weighted", "df_docs", "doc_count_N", "df_ratio",
        "example_repo_name", "example_episode_index", "example_commit_sha"
    ]
    cols = [c for c in cols if c in top.columns]
    top = top[cols]

    top.to_csv(CAND_OUT, index=False, encoding="utf-8")
    print(f"[ok] wrote top-{TOP_K_PER_BOUNDARY} candidates PER boundary: {CAND_OUT}")
    print(f"[info] rows_out={len(top)} (max={(TOP_K_PER_BOUNDARY * df['boundary'].nunique())})")
    if len(top) == 0:
        print("[warn] output is empty — filters may be too strict (try lowering MIN_DF_DOCS or increasing GENERIC_MAX).")


if __name__ == "__main__":
    main()


[ok] wrote top-20 candidates PER boundary: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention\Output\dictionary_candidate_suggestions_TOP20_PER_BOUNDARY.csv
[info] rows_out=60 (max=60)


C:\Users\gilla\AppData\Local\Temp\ipykernel_51148\4054444792.py:141: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: select_top_k(g, k=TOP_K_PER_BOUNDARY))
